# Phase 2a Scaling: Powering the Full-Space Question

Phase 2b asked whether audio carries enough information to recover
`style_ttl`'s perturbation, and got a null result: within-family-residual
R^2 near zero at every perturbation size. That null was an artifact, not a
finding. Ridge regression predicts the test set as a linear combination of
**training target rows**, so its predictions span at most `n_train`
dimensions no matter how good the audio features are. Phase 2b had 240
training samples against a 6,144-dimensional target (24 active
`style_ttl` rows x 256), so the oracle ceiling -- the best any linear
combination of those 240 rows could possibly score -- was capped near 4%
(`py/results/phase2a/ceiling_audit.json`, `eps0.20`: oracle_resid_r2 =
0.0237, achieved = -0.0169). The probe was never given the rank to express
the target it was scored against.

**Phase 2a re-asked the question with a target the estimator can express**,
by confining perturbations to a fixed, nested K-dimensional linear subspace
of the tangent space (`py/phase2b_generate_subspace.py`). With the target
shrunk to match the estimator's rank, audio does carry the perturbation:

| K | n_train | achieved R^2 | oracle ceiling |
|---|---|---|---|
| 4 | 240 | 0.912 | 1.0 |
| 16 | 240 | 0.608 | 1.0 |
| 64 | 480 | 0.232 | 1.0 |

(eps=0.20, `py/results/phase2a/subspace_probe.json`.) Clean controls back
this up: a Hewitt & Liang shuffled-target control and a loudness-only
control both sit near zero at every K, and an amplitude-matched control
shows the decline with K is mostly per-direction signal-to-noise, not a
hard dimensional wall.

**The K=64 scaling sweep (`py/results/phase2a/scaling.json`) is still
rising with no saturation** through the largest n_train on disk:

| n_train | achieved R^2 (mean) | oracle ceiling |
|---|---|---|
| 120 | 0.005 | 1.000 |
| 240 | 0.116 | 1.000 |
| 360 | 0.181 | 1.000 |
| 480 | 0.232 | 1.000 |

The rank-clean log-linear trend slope is 0.163 per log(n); the last
observed step (360 -> 480) has slope 0.177 -- if anything *steeper* than
the global trend, the opposite of what saturation would look like.
Extrapolating that trend to R^2 ~ 0.5 lands at n_train ~ 2,500. **Treat
that number as a rough sketch, not a prediction**: it is a straight line
through four points, extrapolated more than 5x past the last one, and this
project has been burned by confident numbers from thin data before (see
CLAUDE.md's calibration note). It is not trustworthy past roughly
n_train ~ 1,000.

The machine that produced these numbers has 4 CPU cores and no GPU, and
generation runs about 1.4 s/render there -- a corpus large enough to
settle either question (does K=64 recovery keep climbing? does the
*unrestricted* 6,144-dim target become learnable once n_train approaches
its dimension?) cannot be built on it in reasonable time.

**The prize this notebook is built to reach:** at n_train around 6,144,
the K-dimensional subspace restriction can be dropped entirely, and
Phase 2b's original question -- can audio predict the full active-row
style tensor, with no subspace shrinkage -- becomes well-posed for the
first time. That takes roughly 8,000 renders, an estimated 3 hours of
CPU time on the machine above. This notebook runs that generation (CPU,
however many cores Colab hands you), then does the one step that machine
could not do at all: **WavLM feature extraction on a GPU**, and finally
runs both the extended K=64 sweep and, for the first time, the
full-space probe.

## References

- [new-plan.md](../new-plan.md) -- Phase 2 design, including "Phase 2:
  Amortized Style Inversion" and the ceiling-audit / subspace-ladder
  reasoning this notebook extends.
- `py/phase2a_ceiling_audit.py` -- the oracle-ceiling machinery (numerical
  rank + orthogonal projection onto row_space(train targets)) that showed
  Phase 2b's null was a rank artifact.
- `py/phase2b_generate_subspace.py` -- builds the nested K-in-{4,16,64}
  tangent-space basis and draws perturbed styles inside it. This notebook
  imports its `build_basis` / `sample_style` rather than re-deriving the
  geometry.
- `py/phase2b_generate.py` -- the unrestricted (full active-row,
  6,144-dim target) isotropic perturbation generator: this notebook's
  "full-space" condition reuses its `sample_perturbed`.
- `py/phase2b_subspace_embed.py`, `py/phase2b_wavlm_embed.py` -- WavLM-large
  feature extraction and RMS level-matching, reused (with a GPU-placement
  wrapper -- see the extraction section) rather than duplicated.
- `py/phase2b_subspace_probe.py`, `py/phase2a_scaling.py`, `py/phase2b_probe.py`
  -- the ridge probe, its train/test id-matching, and the n_train sweep
  logic this notebook extends to the larger corpus.
- [docs/style_extraction_colab.ipynb](style_extraction_colab.ipynb) and
  [docs/presentation_axis_colab.ipynb](presentation_axis_colab.ipynb) --
  this notebook's siblings; it follows their asset-fetching and
  checkpointing conventions.

Run this on a GPU runtime (Runtime > Change runtime type > T4 or better).
**Only the WavLM extraction stage uses the GPU.** Style generation is
`onnxruntime` on CPU throughout -- see the generation section for why, in
detail; the short version is that `py/helper.py`'s `load_text_to_speech`
raises `NotImplementedError("GPU mode is not fully tested")` for
`use_gpu=True`, and this notebook does not route around that guard for
the corpus that becomes this project's evidence base (it *does* run one
honest, clearly-labelled timing probe of the CUDA execution provider, see
below, so the "GPU doesn't help generation" claim is measured here rather
than only asserted).

In [ ]:
# 1. GPU check. WavLM extraction (stage 4) needs it; style generation (stage
# 3) does not and will not use it -- see the markdown above and the
# onnxruntime-gpu timing cell in stage 3 for why.
import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        'No CUDA device. Runtime > Change runtime type > Hardware accelerator: '
        'GPU (T4 or better), then Runtime > Run all. WavLM feature extraction '
        '(stage 4) is the reason this notebook needs a GPU runtime at all -- '
        'generation itself is CPU-bound regardless.')

print('GPU: %s' % torch.cuda.get_device_name(0))
print('VRAM: %.1f GB' % (torch.cuda.get_device_properties(0).total_memory / 1024 ** 3))
print('torch: %s, CUDA: %s' % (torch.__version__, torch.version.cuda))

import os
N_CPUS = os.cpu_count() or 1
print('CPU cores visible to this runtime: %d' % N_CPUS)
print('(Style generation, stage 3, uses all of them; note this number for '
      'the time estimate in the config cell below.)')

In [ ]:
# 2. Drive: the corpus, features and results all live here so a disconnect
# costs minutes of resync, not hours of regeneration.
from pathlib import Path

from google.colab import drive

drive.mount('/content/drive')

WORKSPACE = Path('/content/drive/MyDrive/supertonic-phase2a-scaling')
GEN_DIR = WORKSPACE / 'subspace'        # K-condition corpus (subspace ladder)
FULL_DIR = WORKSPACE / 'fullspace'      # unrestricted-target corpus
RESULTS = WORKSPACE / 'results'
LOGS = WORKSPACE / 'logs'
for d in (WORKSPACE, GEN_DIR, GEN_DIR / 'audio16k', FULL_DIR, FULL_DIR / 'audio16k',
          RESULTS, LOGS):
    d.mkdir(parents=True, exist_ok=True)

print('Workspace: %s' % WORKSPACE)

In [ ]:
# 3. Assets and the fork's scripts. snapshot_download rather than git-lfs for
# the ONNX graphs -- see docs/style_extraction_colab.ipynb's cell 4: git-lfs
# over 392 MB is flaky in Colab and the resume logic here is the point. The
# fork is cloned at the branch carrying the phase2 scripts (they are not on
# upstream); set REPO_URL / REPO_REF to your own fork+branch if you are
# running from somewhere else.
import subprocess
import sys

REPO = Path('/content/supertonic')
REPO_URL = 'https://github.com/supertone-inc/supertonic.git'   # set to your fork
REPO_REF = 'claude/roadmap-next-steps-tx1aa8'                  # branch carrying py/phase2*.py

if not REPO.exists():
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', REPO_REF,
                    REPO_URL, str(REPO)], check=True)
REPO_PY = str(REPO / 'py')
if REPO_PY not in sys.path:
    sys.path.insert(0, REPO_PY)

needed = ['helper.py', 'phase2b_generate.py', 'phase2b_generate_subspace.py',
          'phase2b_subspace_embed.py', 'phase2b_subspace_probe.py',
          'phase2a_scaling.py', 'phase2b_probe.py', 'phase2a_ceiling_audit.py',
          'phase2b_wavlm_embed.py', 'phase2b_wavlm.py']
missing = [n for n in needed if not (REPO / 'py' / n).exists()]
if missing:
    raise FileNotFoundError(
        '%s missing from %s -- REPO_URL/REPO_REF do not point at a checkout '
        'carrying the phase2 scripts. Point REPO_REF at the branch that has '
        'them and rerun this cell.' % (missing, REPO_PY))
print('Repo scripts present: %s' % REPO_PY)

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'onnxruntime==1.23.1', 'numpy>=1.26.0', 'soundfile>=0.12.1',
                'librosa>=0.10.0', 'PyYAML>=6.0', 'huggingface_hub', 'scipy',
                'scikit-learn', 'torch', 'torchaudio'], check=True)

from huggingface_hub import snapshot_download

ASSETS = Path('/content/supertonic-assets')
snapshot_download(repo_id='Supertone/supertonic-3', local_dir=str(ASSETS),
                  allow_patterns=['onnx/*', 'voice_styles/*'])
ONNX_DIR = str(ASSETS / 'onnx')
VOICE_STYLE_DIR = str(ASSETS / 'voice_styles')
missing_assets = [n for n in ('duration_predictor.onnx', 'text_encoder.onnx',
                              'vector_estimator.onnx', 'vocoder.onnx', 'tts.json',
                              'unicode_indexer.json')
                  if not (Path(ONNX_DIR) / n).exists()]
if missing_assets:
    raise FileNotFoundError('Missing ONNX assets: %s' % missing_assets)
print('ONNX: %s' % ONNX_DIR)
print('Presets: %s' % sorted(p.stem for p in Path(VOICE_STYLE_DIR).glob('*.json')))

## Does `onnxruntime-gpu` help generation? Measured, not assumed

`py/helper.py`'s `load_text_to_speech(onnx_dir, use_gpu=True)` raises
`NotImplementedError("GPU mode is not fully tested")` -- upstream never
finished validating the GPU path. This notebook does not treat that as
license to route around it silently. The cell below builds a
`TextToSpeech` two ways, using the same lower-level `helper.load_onnx_all`
the guarded wrapper itself calls, so nothing here bypasses anything the
repo doesn't already expose as a building block:

- **CPU**: `providers=["CPUExecutionProvider"]`, exactly what
  `load_text_to_speech` does today.
- **GPU, if `onnxruntime-gpu` installs and a CUDA provider is available**:
  `providers=["CUDAExecutionProvider", "CPUExecutionProvider"]`.

Both render the same handful of clips and are timed. Whichever is not
faster is dropped, and the notebook says so in plain language before
stage 3 runs. If the GPU path *is* faster, it is still labelled
unvalidated in every output that uses it (this project has not confirmed
numerical agreement between the two providers for this graph) -- read the
result before trusting it for anything beyond wall-clock time.

In [ ]:
# 3b. Time CPU vs (if available) CUDA generation on a handful of clips.
import time

import numpy as np

import helper

N_TIMING_CLIPS = 6
TIMING_TEXT = 'The quick brown fox jumps over the lazy dog.'

def build_tts(providers, threads=None):
    opts = helper.ort.SessionOptions()
    if threads:
        opts.intra_op_num_threads = threads
    cfgs = helper.load_cfgs(ONNX_DIR)
    dp_ort, text_enc_ort, vector_est_ort, vocoder_ort = helper.load_onnx_all(
        ONNX_DIR, opts, providers)
    text_processor = helper.load_text_processor(ONNX_DIR)
    return helper.TextToSpeech(cfgs, text_processor, dp_ort, text_enc_ort,
                               vector_est_ort, vocoder_ort)

def time_provider(providers, label):
    try:
        tts = build_tts(providers)
    except Exception as exc:
        print('[%s] unavailable: %r' % (label, exc))
        return None
    base_style = helper.load_voice_style([str(Path(VOICE_STYLE_DIR) / 'M1.json')])
    t0 = time.time()
    for i in range(N_TIMING_CLIPS):
        np.random.seed(i)
        tts(TIMING_TEXT, 'en', base_style, 8, 1.05)
    elapsed = time.time() - t0
    per_clip = elapsed / N_TIMING_CLIPS
    print('[%s] %d clips in %.2fs -> %.3f s/clip' % (label, N_TIMING_CLIPS, elapsed, per_clip))
    return per_clip

cpu_spc = time_provider(['CPUExecutionProvider'], 'CPU')

gpu_spc = None
try:
    # onnxruntime and onnxruntime-gpu both install to the same top-level
    # 'onnxruntime' package name and can leave a broken mixed install if the
    # CPU wheel is not removed first.
    subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', '-q', 'onnxruntime'],
                   check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'onnxruntime-gpu==1.23.1'],
                   check=True)
    import importlib
    importlib.reload(helper)  # helper imports onnxruntime at module load; needs a fresh
                              # look at the environment after swapping the cpu/gpu build
    if 'CUDAExecutionProvider' in helper.ort.get_available_providers():
        gpu_spc = time_provider(['CUDAExecutionProvider', 'CPUExecutionProvider'], 'CUDA')
    else:
        print('onnxruntime-gpu installed but CUDAExecutionProvider is not in '
              'get_available_providers(); no CUDA-capable onnxruntime build is active.')
except Exception as exc:
    print('onnxruntime-gpu install/timing failed: %r' % exc)
finally:
    if gpu_spc is None:
        # Either the swap failed outright or CUDA wasn't usable once swapped.
        # Guarantee a working CPU onnxruntime for every cell after this one --
        # generation absolutely must work regardless of how this probe went.
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'onnxruntime==1.23.1'],
                       check=True)
        import importlib
        importlib.reload(helper)
        assert 'CPUExecutionProvider' in helper.ort.get_available_providers()

USE_GPU_FOR_GENERATION = bool(gpu_spc is not None and cpu_spc is not None and gpu_spc < cpu_spc)
GEN_S_PER_CLIP_MEASURED = min(x for x in (cpu_spc, gpu_spc) if x is not None)

if USE_GPU_FOR_GENERATION:
    print('\nCUDAExecutionProvider measured faster (%.3fs vs %.3fs/clip) and will be used '
          'for generation below. This provider is unvalidated for this graph beyond wall-clock '
          'time -- treat generated audio as provisional pending a spot numerical check against '
          'the CPU path.' % (gpu_spc, cpu_spc))
else:
    print('\nGPU either unavailable, not faster, or untested cleanly -- generation below runs '
          'on CPU, as py/helper.py already defaults to. Measured %.3f s/clip on this instance.'
          % GEN_S_PER_CLIP_MEASURED)

## Configuration

Every knob that changes how much this notebook does or how long it takes
lives here. Nothing below this cell needs manual edits for a top-to-bottom
run.

- **`N_TOTAL`** -- how many K-subspace clips to generate (in addition to
  whatever a resumed run already has on Drive). Default 8,000: enough to
  push the K=64 training pool from 480 (Phase 2a's ceiling) to roughly
  6,000, which is where the extrapolated "R^2 ~ 0.5 near n_train ~ 2,500"
  guess gets tested against real data instead of a 5x extrapolation.
- **`K`** -- which subspace dimension to generate at. Default 64, the
  condition whose scaling curve had not saturated. (K=4 and K=16 already
  have oracle ceilings of 1.0 at n_train=240 and achieved R^2 of 0.91/0.61
  there -- there is little open question left at those K to spend renders
  on.)
- **`EPS`** -- perturbation magnitude, matched to Phase 2a/2b's 0.20 so
  results are comparable.
- **`GENERATE_FULL_SPACE`** / **`N_FULL`** -- whether to also generate an
  *unrestricted* condition: no subspace projection, isotropic perturbation
  across all 24 active rows (`py/phase2b_generate.py`'s own model), target
  dimension the full 6,144. This is Phase 2b's original, never-adequately-
  powered experiment. Default on, `N_FULL = 8000` (n_train ~ 6,000 at the
  default 0.25 test fraction) -- close to the target's own dimensionality,
  which is the point.
- **`N_WORKERS`** -- process pool size for generation. Defaults to
  `os.cpu_count()`; each worker gets its own single-threaded onnxruntime
  session so the pool doesn't oversubscribe the machine.

**Time estimate below is honest about its basis**: the 1.4 s/clip figure
in the opening markdown was measured on a *different* machine (4 CPU
cores, no GPU) during Phase 2a's original CPU-only run. This cell's
estimate starts from whichever of that figure or this Colab instance's own
measured CPU/GPU generation time (previous cell) is available, scaled by
`N_CPUS` workers under an optimistic linear-speedup assumption, and is
labelled as an estimate throughout -- the generation cell re-measures
actual throughput and corrects the printed ETA as it runs.

In [ ]:
# 4. Configuration.
N_TOTAL = 8000            # additional K-subspace clips to generate (resumable)
K = 64                    # subspace dimension
EPS = 0.20                # perturbation magnitude, matches Phase 2a/2b
TEST_FRAC = 0.25          # fraction of N_TOTAL held out as test, same convention as
                          # phase2b_generate_subspace.py

GENERATE_FULL_SPACE = True
N_FULL = 8000             # unrestricted-target clips (only used if GENERATE_FULL_SPACE)

N_WORKERS = N_CPUS         # process-pool size for generation (stage 3)
RENDER_THREADS_PER_WORKER = 1   # keeps N_WORKERS x this <= N_CPUS

if USE_GPU_FOR_GENERATION:
    # There is exactly one GPU. N_CPUS worker processes each opening their own
    # CUDAExecutionProvider session against it would contend for VRAM and the
    # CUDA driver rather than parallelize -- cap the pool low instead. This
    # path is only reachable if the timing cell above measured CUDA as
    # genuinely faster than CPU for this graph on this instance.
    N_WORKERS = min(N_WORKERS, 2)
    print('USE_GPU_FOR_GENERATION is True: capping N_WORKERS to %d (one GPU, not %d cores) -- '
          'see the timing cell above for why GPU was selected at all.' % (N_WORKERS, N_CPUS))

REFERENCE_S_PER_CLIP = 1.4   # measured on a 4-core, no-GPU machine (Phase 2a's own run);
                              # NOT measured on this Colab instance -- see stage-3 markdown
this_machine_s_per_clip = GEN_S_PER_CLIP_MEASURED  # from the timing cell just above

est_total_clips = N_TOTAL + (N_FULL if GENERATE_FULL_SPACE else 0)
est_serial_hours = est_total_clips * this_machine_s_per_clip / 3600.0
est_parallel_hours = est_serial_hours / max(1, N_WORKERS)

print('Planned: %d K=%d clips%s' % (
    N_TOTAL, K, (' + %d full-space clips' % N_FULL) if GENERATE_FULL_SPACE else ''))
print('Measured on this instance: %.3f s/clip (%s), %d workers planned'
      % (this_machine_s_per_clip, 'GPU/CUDA' if USE_GPU_FOR_GENERATION else 'CPU', N_WORKERS))
print('ESTIMATE (not measured): %.1f clip-hours serial -> ~%.1f h wall clock at %dx '
      'parallelism, IF speedup is linear -- it typically is not, onnxruntime CPU inference '
      'often scales sublinearly past 4-8 workers. Treat this as an upper bound on speed, i.e. '
      'a LOWER bound on time.' % (est_serial_hours, est_parallel_hours, N_WORKERS))
print('The generation cell below re-measures actual throughput every ~100 clips and prints '
      'a corrected ETA from real data instead of this estimate.')

## Stage 3: generation (CPU-bound -- this is the long pole)

**Be honest about what this stage is.** Every clip is one forward pass
through four frozen ONNX graphs -- `duration_predictor`, `text_encoder`,
`vector_estimator`, `vocoder` -- run through `onnxruntime`'s
`CPUExecutionProvider`. The timing cell above already checked whether
`onnxruntime-gpu` changes that; unless it reported CUDA as measurably
faster, this stage runs on CPU exactly as `py/helper.py` already defaults
to, and the GPU this notebook's runtime is paying for sits idle until
stage 4.

**What this stage does, in two parts, both resumable:**

1. **Precompute the style tensors** (cheap: pure numpy, no audio). For the
   K-subspace condition this reuses `phase2b_generate_subspace.build_basis`
   and `.sample_style` verbatim -- same tangent-space geometry, same QR
   nesting, same per-row renormalization -- so this corpus's basis is
   constructed exactly the way Phase 2a's was, just with more samples
   drawn from it. For the full-space condition it reuses
   `phase2b_generate.sample_perturbed` verbatim -- the same isotropic
   per-row perturbation Phase 2b used, just at a larger `N_FULL`.
2. **Render audio in parallel** across `N_WORKERS` processes, each holding
   its own single-threaded `onnxruntime` session (so the pool doesn't
   oversubscribe the machine the way `N_WORKERS` sessions each defaulting
   to "use every core" would). A small worker module is written to Drive
   first (`gen_worker.py`, same pattern as
   `docs/style_extraction_colab.ipynb` writing `style_tools.py`) so the
   process pool can use the `spawn` start method safely -- `fork` after a
   CUDA context exists in the parent (this notebook's own GPU check) is a
   known source of hangs in child processes, and `spawn` needs a real,
   importable module file rather than a function defined in a notebook
   cell.

**Checkpointing**: every rendered clip is one file
(`K64_00000.wav`, ...). Resuming a disconnected session means: recompute
the (deterministic, cheap) style tensors, then skip any index whose wav
file already exists on Drive -- read it back to recover its stats
(peak/rms/duration) rather than re-rendering. 8,000+ renders at an hour or
more of wall-clock time will not reliably fit one Colab session; this is
what makes that survivable.

In [ ]:
# 5. Write the worker module. A real file, not a notebook-cell function --
# ProcessPoolExecutor with the 'spawn' start method (used below to avoid
# forking after this notebook's own CUDA initialization) pickles workers by
# module + name, and Jupyter's __main__ does not resolve for that.
WORKER_SRC = '''
import numpy as np
import soundfile as sf
from scipy.signal import resample_poly

import sys
sys.path.insert(0, %r)
import helper
from phase2b_generate import Style, LANG, TOTAL_STEP, SPEED, TEXTS, EMBED_SR

_tts = None
_dp_ref = None


def init_worker(onnx_dir, dp_ref_list, threads, use_cuda):
    # Runs once per worker process (spawn start method): build one
    # single-threaded onnxruntime session so N_WORKERS processes together use
    # about N_WORKERS threads, not N_WORKERS x all-cores.
    global _tts, _dp_ref
    opts = helper.ort.SessionOptions()
    opts.intra_op_num_threads = threads
    opts.inter_op_num_threads = 1
    providers = (["CUDAExecutionProvider", "CPUExecutionProvider"] if use_cuda
                 else ["CPUExecutionProvider"])
    cfgs = helper.load_cfgs(onnx_dir)
    dp_ort, text_enc_ort, vector_est_ort, vocoder_ort = helper.load_onnx_all(
        onnx_dir, opts, providers)
    text_processor = helper.load_text_processor(onnx_dir)
    _tts = helper.TextToSpeech(cfgs, text_processor, dp_ort, text_enc_ort,
                               vector_est_ort, vocoder_ort)
    _dp_ref = np.asarray(dp_ref_list, dtype=np.float32)[None, :, :]


def render_one(job):
    # job: {idx, ttl (50x256 nested list), text_idx, seed, out_path}. Returns
    # the manifest stats fields for this clip. Independent of every other job
    # -- dispatch order does not matter, only (ttl, text_idx, seed) do, which
    # is why this is safe to parallelize even though the *drawing* of styles
    # upstream is a strictly sequential RNG sequence.
    ttl = np.asarray(job["ttl"], dtype=np.float32)[None, :, :]
    np.random.seed(job["seed"])   # vocoder latent RNG, as in phase2b_generate*.py
    wav, dur = _tts(TEXTS[job["text_idx"]], LANG, Style(ttl, _dp_ref.copy()),
                    TOTAL_STEP, SPEED)
    trimmed = wav[0, : int(_tts.sample_rate * dur[0].item())].astype(np.float32)
    w16 = resample_poly(trimmed, EMBED_SR, _tts.sample_rate).astype(np.float32)
    sf.write(job["out_path"], w16, EMBED_SR, subtype="PCM_16")
    return {
        "idx": job["idx"],
        "duration_sec": float(dur[0].item()),
        "peak": float(np.abs(trimmed).max()),
        "rms": float(np.sqrt((trimmed.astype(np.float64) ** 2).mean())),
        "finite": bool(np.isfinite(trimmed).all()),
    }


def stats_from_existing(path, embed_sr):
    # Recompute the manifest stats fields from an already-rendered 16k wav,
    # for a resumed clip this process does not re-render. duration here is
    # frames/embed_sr, a slightly different definition from a freshly
    # rendered clip's tts.sample_rate-based duration; nothing downstream
    # depends on the distinction (only idx/K/split/text_idx/seed key the
    # probe's id-matching).
    wav, sr = sf.read(path, dtype="float32")
    assert sr == embed_sr, (path, sr)
    return {
        "duration_sec": float(len(wav) / embed_sr),
        "peak": float(np.abs(wav).max()) if len(wav) else 0.0,
        "rms": float(np.sqrt((wav.astype(np.float64) ** 2).mean())) if len(wav) else 0.0,
        "finite": bool(np.isfinite(wav).all()),
    }
''' % (REPO_PY,)

WORKER_MODULE_PATH = WORKSPACE / 'gen_worker.py'
WORKER_MODULE_PATH.write_text(WORKER_SRC)
WORKER_DIR = str(WORKSPACE)
if WORKER_DIR not in sys.path:
    sys.path.insert(0, WORKER_DIR)
import gen_worker
import importlib
importlib.reload(gen_worker)
print('Wrote %s' % WORKER_MODULE_PATH)

In [ ]:
# 6. Precompute style tensors for the K-subspace condition. Pure numpy, no
# audio -- cheap enough to always recompute at the start of a session rather
# than checkpoint separately; what IS checkpointed is the audio (next cell).
from phase2b_generate import ACTIVE_ROWS, SEED_BASE, TEXTS
from phase2b_generate_subspace import BASE_PRESET, BASIS_SEED, build_basis, sample_style
from helper import load_voice_style

base_style = load_voice_style([str(Path(VOICE_STYLE_DIR) / (BASE_PRESET + '.json'))])
base_ttl = base_style.ttl.astype(np.float32)         # (1, 50, 256)
dp_ref = base_style.dp.copy()[0]                      # (8, 16), held fixed

B, P = build_basis(base_ttl)
ortho_err = float(np.abs(B @ B.T - np.eye(B.shape[0])).max())
print('orthonormality check: max |B B^T - I| = %.3e' % ortho_err)
assert ortho_err < 1e-5

N_TEST = max(1, int(round(N_TOTAL * TEST_FRAC)))
N_TRAIN = N_TOTAL - N_TEST
splits = ['train'] * N_TRAIN + ['test'] * N_TEST
SAMPLE_SEED = SEED_BASE + 1 + K   # offset by K so different K choices draw independent samples
sample_rng = np.random.default_rng(SAMPLE_SEED)
sample_rng.shuffle(splits)

subspace_jobs = []
c_drawn_all = np.zeros((N_TOTAL, K), dtype=np.float32)
c_realized_all = np.zeros((N_TOTAL, K), dtype=np.float32)
ttl_all = np.zeros((N_TOTAL, 50, 256), dtype=np.float32)
in_subspace_fractions = []

for local_idx in range(N_TOTAL):
    rows, c_drawn, c_realized, frac = sample_style(sample_rng, B, P, K, EPS)
    ttl = base_ttl.copy()
    ttl[0, ACTIVE_ROWS, :] = rows.astype(np.float32)
    text_i = int(sample_rng.integers(len(TEXTS)))
    seed = SEED_BASE + 10_000_000 + local_idx     # offset clear of phase2b_generate*'s own range
    subspace_jobs.append({
        'idx': local_idx, 'K': K, 'text_idx': text_i, 'seed': seed,
        'split': splits[local_idx], 'in_subspace_fraction': frac,
        'ttl': ttl[0].tolist(),
        'file': 'K%d_%05d.wav' % (K, local_idx),
    })
    c_drawn_all[local_idx] = c_drawn
    c_realized_all[local_idx] = c_realized
    ttl_all[local_idx] = ttl[0]
    in_subspace_fractions.append(frac)

print('%d K=%d samples drawn (%d train / %d test). in_subspace_fraction mean %.4f, min %.4f'
      % (N_TOTAL, K, N_TRAIN, N_TEST, float(np.mean(in_subspace_fractions)),
         float(np.min(in_subspace_fractions))))

np.savez_compressed(GEN_DIR / 'subspace.npz', basis=B.astype(np.float32), base_ttl=base_ttl[0],
                    active_rows=np.array(ACTIVE_ROWS), **{
                        'c_drawn_K%d' % K: c_drawn_all,
                        'c_realized_K%d' % K: c_realized_all,
                        'ttl_K%d' % K: ttl_all,
                    })
print('Wrote %s' % (GEN_DIR / 'subspace.npz'))

In [ ]:
# 7. Precompute style tensors for the full-space condition (if enabled). Same
# isotropic per-row perturbation phase2b_generate.py itself uses -- no
# subspace projection, target is the full 6,144-dim active-row block.
full_jobs = []
if GENERATE_FULL_SPACE:
    from phase2b_generate import sample_perturbed, TEXTS as _TEXTS

    n_test_full = max(1, int(round(N_FULL * TEST_FRAC)))
    n_train_full_planned = N_FULL - n_test_full
    splits_full = ['train'] * n_train_full_planned + ['test'] * n_test_full
    full_sample_seed = SEED_BASE + 2 + K
    full_rng = np.random.default_rng(full_sample_seed)
    full_rng.shuffle(splits_full)

    ttl_full_all = np.zeros((N_FULL, 50, 256), dtype=np.float32)
    for local_idx in range(N_FULL):
        ttl = sample_perturbed(full_rng, base_ttl, EPS)
        text_i = int(full_rng.integers(len(_TEXTS)))
        seed = SEED_BASE + 20_000_000 + local_idx
        full_jobs.append({
            'idx': local_idx, 'text_idx': text_i, 'seed': seed,
            'split': splits_full[local_idx], 'ttl': ttl[0].tolist(),
            'file': 'full_%05d.wav' % local_idx,
        })
        ttl_full_all[local_idx] = ttl[0]

    np.savez_compressed(FULL_DIR / 'styles.npz', ttl=ttl_full_all,
                        active_rows=np.array(ACTIVE_ROWS), base_ttl=base_ttl[0])
    print('%d full-space samples drawn (%d train / %d test).'
          % (N_FULL, n_train_full_planned, n_test_full))
    print('Wrote %s' % (FULL_DIR / 'styles.npz'))
else:
    print('GENERATE_FULL_SPACE is False -- skipping. The results cell will report the '
          'K=%d sweep only; the full-space (Phase 2b-original) probe needs this stage.' % K)

In [ ]:
# 8. Render, in parallel, with per-clip checkpointing. Runs both conditions'
# pending jobs through one process pool. A clip whose wav already exists is
# skipped and its stats read back rather than re-rendered.
import concurrent.futures as cf
import multiprocessing as mp
import time as _time

def _plan_pending(jobs, audio_dir):
    pending, done_stats = [], {}
    for job in jobs:
        out_path = str(Path(audio_dir) / job['file'])
        if Path(out_path).exists():
            try:
                done_stats[job['idx']] = gen_worker.stats_from_existing(out_path, 16000)
                continue
            except Exception as exc:
                print('  [redo] %s unreadable (%r), re-rendering' % (job['file'], exc))
        job = dict(job, out_path=out_path)
        pending.append(job)
    return pending, done_stats

sub_pending, sub_done = _plan_pending(subspace_jobs, GEN_DIR / 'audio16k')
full_pending, full_done = _plan_pending(full_jobs, FULL_DIR / 'audio16k') if full_jobs else ([], {})

print('K=%d: %d/%d already rendered, %d pending' % (K, len(sub_done), len(subspace_jobs), len(sub_pending)))
if full_jobs:
    print('full-space: %d/%d already rendered, %d pending' % (len(full_done), len(full_jobs), len(full_pending)))

all_pending = sub_pending + full_pending
render_stats = {'subspace': dict(sub_done), 'full': dict(full_done)}

if all_pending:
    ctx = mp.get_context('spawn')
    started = _time.time()
    n_done = 0
    with cf.ProcessPoolExecutor(max_workers=N_WORKERS, mp_context=ctx,
                                initializer=gen_worker.init_worker,
                                initargs=(ONNX_DIR, dp_ref.tolist(), RENDER_THREADS_PER_WORKER,
                                          USE_GPU_FOR_GENERATION)) as pool:
        futures = {}
        for job in all_pending:
            tag = 'full' if job['file'].startswith('full_') else 'subspace'
            fut = pool.submit(gen_worker.render_one, job)
            futures[fut] = tag

        for fut in cf.as_completed(futures):
            tag = futures[fut]
            stats = fut.result()
            render_stats[tag][stats['idx']] = stats
            n_done += 1
            if n_done % 100 == 0 or n_done == len(all_pending):
                elapsed = _time.time() - started
                rate = elapsed / n_done
                remaining = (len(all_pending) - n_done) * rate
                print('  %d/%d rendered  %.3fs/clip (measured)  ~%.1f min remaining'
                      % (n_done, len(all_pending), rate, remaining / 60.0), flush=True)
    print('\nRendering complete: %d clips this run.' % n_done)
else:
    print('Nothing pending -- every planned clip already exists on Drive.')

bad_sub = [i for i, s in render_stats['subspace'].items() if not s['finite'] or s['peak'] >= 1.0]
print('subspace: non-finite or clipped clips: %d/%d' % (len(bad_sub), len(render_stats['subspace'])))
if full_jobs:
    bad_full = [i for i, s in render_stats['full'].items() if not s['finite'] or s['peak'] >= 1.0]
    print('full-space: non-finite or clipped clips: %d/%d' % (len(bad_full), len(render_stats['full'])))

In [ ]:
# 9. Assemble manifest.json for the K-subspace condition, in the same schema
# phase2b_generate_subspace.py itself writes, so py/phase2b_subspace_probe.py
# and py/phase2a_scaling.py can also be run against this corpus unmodified
# from the command line if you want a second, independent check.
import datetime
import json as _json

sub_records = []
for job in subspace_jobs:
    stats = render_stats['subspace'][job['idx']]
    sub_records.append({
        'K': job['K'], 'idx': job['idx'], 'file': job['file'],
        'text_idx': job['text_idx'], 'seed': job['seed'], 'split': job['split'],
        'in_subspace_fraction': job['in_subspace_fraction'],
        'duration_sec': stats['duration_sec'], 'peak': stats['peak'], 'rms': stats['rms'],
        'finite': stats['finite'],
    })

sub_manifest = {
    'experiment': 'phase2a_scaling_colab_subspace',
    'date': datetime.date.today().isoformat(),
    'base_preset': BASE_PRESET, 'k_list': [K], 'n_per_k': {str(K): N_TOTAL},
    'n_total': N_TOTAL, 'eps': EPS, 'active_rows': ACTIVE_ROWS,
    'lang': 'en', 'total_step': 8, 'speed': 1.05,
    'style_dp': "held fixed at %s's for every sample" % BASE_PRESET,
    'embed_sample_rate': 16000, 'basis_seed': BASIS_SEED, 'sample_seed': SAMPLE_SEED,
    'orthonormality_off_diag_max': ortho_err,
    'generated_by': 'docs/phase2a_scaling_colab.ipynb',
    'records': sub_records,
}
with open(GEN_DIR / 'manifest.json', 'w') as f:
    _json.dump(sub_manifest, f, indent=2)
print('Wrote %s (%d records)' % (GEN_DIR / 'manifest.json', len(sub_records)))

if full_jobs:
    full_records = []
    for job in full_jobs:
        stats = render_stats['full'][job['idx']]
        full_records.append({
            'idx': job['idx'], 'file': job['file'], 'text_idx': job['text_idx'],
            'seed': job['seed'], 'split': job['split'],
            'duration_sec': stats['duration_sec'], 'peak': stats['peak'], 'rms': stats['rms'],
            'finite': stats['finite'],
        })
    full_manifest = {
        'experiment': 'phase2a_scaling_colab_fullspace',
        'date': datetime.date.today().isoformat(),
        'base_preset': BASE_PRESET, 'n_total': N_FULL, 'eps': EPS,
        'active_rows': ACTIVE_ROWS, 'lang': 'en', 'total_step': 8, 'speed': 1.05,
        'embed_sample_rate': 16000, 'sample_seed': full_sample_seed,
        'generated_by': 'docs/phase2a_scaling_colab.ipynb',
        'records': full_records,
    }
    with open(FULL_DIR / 'manifest.json', 'w') as f:
        _json.dump(full_manifest, f, indent=2)
    print('Wrote %s (%d records)' % (FULL_DIR / 'manifest.json', len(full_records)))

## Stage 4: WavLM extraction -- this is where the GPU pays off

`py/phase2b_wavlm_embed.py`'s `pooled_features` builds its input tensor on
CPU (`torch.from_numpy(wav)[None, :]`, no `.to(device)`) and never moves the
model, so importing it as-is always runs on whatever device the model
happens to be on -- CPU, by construction, since `load_model()` never calls
`.to('cuda')` either. On the machine that produced Phase 2a's numbers this
ran at roughly 1.0 s/clip.

Below is a **minimal, GPU-placing wrapper**, not a reimplementation: it
reuses `load_model` and the constants `DIM` / `N_LAYERS` verbatim from
`phase2b_wavlm_embed`, and the forward pass itself
(`model.extract_features`, per-layer mean/std pooling) is line-for-line the
same as `pooled_features` -- the only change is `.to(device)` on the input
tensor and moving the model once. Before trusting a GPU-extracted feature
for anything, the cell cross-checks it against the unmodified CPU
`pooled_features` on one clip and asserts they agree to float32 tolerance,
in keeping with this project's rule that a mirror of existing logic gets a
numeric cross-check, not just an assertion that it "should" match.

RMS level-matching before extraction reuses `phase2b_subspace_embed.level_match`
verbatim (target RMS 0.05, silence-skip threshold), so loudness cannot pose
as a real feature difference downstream -- see that script's docstring and
CLAUDE.md's "level-match before listening/before treating a distance as
evidence" note.

In [ ]:
# 10. GPU-placed WavLM-large extraction, cross-checked against the CPU
# reference implementation on one clip before trusting the batch.
import soundfile as sf
import torch as _torch

from phase2b_subspace_embed import TARGET_RMS, level_match
from phase2b_wavlm_embed import DIM, N_LAYERS, load_model, pooled_features

DEVICE = 'cuda' if _torch.cuda.is_available() else 'cpu'
bundle, wavlm_model = load_model()
wavlm_model = wavlm_model.to(DEVICE)
NORMALIZE = bool(getattr(bundle, '_normalize_waveform', True))
print('WavLM-large on %s; normalize_waveform=%s' % (DEVICE, NORMALIZE))


def pooled_features_gpu(model, normalize, wav_array, device):
    # Same forward pass and pooling as phase2b_wavlm_embed.pooled_features,
    # operating on an in-memory leveled waveform instead of a file path (skips
    # a redundant WAV encode/decode round trip) and placing the tensor on
    # `device` before the forward pass.
    x = _torch.from_numpy(wav_array)[None, :].to(device)
    if normalize:
        x = _torch.nn.functional.layer_norm(x, x.shape)
    with _torch.no_grad():
        feats, _ = model.extract_features(x)
    assert len(feats) == N_LAYERS, len(feats)
    mean = np.empty((N_LAYERS, DIM), dtype=np.float32)
    std = np.empty((N_LAYERS, DIM), dtype=np.float32)
    for li, f in enumerate(feats):
        f = f[0]
        mean[li] = f.mean(0).cpu().numpy()
        std[li] = f.std(0).cpu().numpy()
    return mean, std


# Cross-check: one clip, GPU wrapper vs. the unmodified CPU function.
_check_path = str(GEN_DIR / 'audio16k' / subspace_jobs[0]['file'])
_wav, _sr = sf.read(_check_path, dtype='float32')
_leveled, _ = level_match(_wav)
_mean_cpu, _std_cpu = pooled_features(load_model()[1], NORMALIZE, _check_path)
_mean_gpu, _std_gpu = pooled_features_gpu(wavlm_model, NORMALIZE, _leveled, DEVICE)
_drift = float(np.abs(_mean_cpu - _mean_gpu).max())
print('GPU-wrapper vs CPU-reference max |mean| drift on one clip: %.3e' % _drift)
assert _drift < 1e-3, 'GPU wrapper disagrees with phase2b_wavlm_embed.pooled_features'
print('Cross-check passed.')

In [ ]:
# 11. Batch-extract WavLM features for both conditions. Reports clips/sec so
# the CPU-vs-GPU payoff claimed above is measured, not asserted.
import soundfile as sf


def embed_condition(records, audio_dir, out_path, has_k):
    if out_path.exists():
        cached = np.load(out_path)
        if int(cached['n'][0]) == len(records):
            print('%s already has all %d clips embedded -- skipping.' % (out_path, len(records)))
            return cached
    mean_buf = np.zeros((len(records), N_LAYERS, DIM), dtype=np.float32)
    std_buf = np.zeros((len(records), N_LAYERS, DIM), dtype=np.float32)
    rms_buf = np.zeros(len(records), dtype=np.float32)
    idx_buf = np.zeros(len(records), dtype=np.int64)
    k_buf = np.full(len(records), -1, dtype=np.int64)
    n_kept = 0
    n_skipped = 0
    t0 = time.time()
    for i, r in enumerate(records):
        wav, sr = sf.read(str(Path(audio_dir) / r['file']), dtype='float32')
        assert sr == 16000
        leveled, rms_pre = level_match(wav)
        if leveled is None:
            n_skipped += 1
            continue
        mean, std = pooled_features_gpu(wavlm_model, NORMALIZE, leveled, DEVICE)
        mean_buf[n_kept] = mean
        std_buf[n_kept] = std
        rms_buf[n_kept] = rms_pre
        idx_buf[n_kept] = int(r['idx'])
        if has_k:
            k_buf[n_kept] = int(r['K'])
        n_kept += 1
        if n_kept % 200 == 0 or (i + 1) == len(records):
            elapsed = time.time() - t0
            print('  %d/%d embedded  %.1f clips/sec' % (n_kept, len(records), n_kept / max(elapsed, 1e-9)),
                  flush=True)
    mean_buf, std_buf = mean_buf[:n_kept], std_buf[:n_kept]
    rms_buf, idx_buf, k_buf = rms_buf[:n_kept], idx_buf[:n_kept], k_buf[:n_kept]
    np.savez_compressed(out_path, mean=mean_buf, std=std_buf, idx=idx_buf, K=k_buf,
                        rms_pre=rms_buf, target_rms=np.array([TARGET_RMS], dtype=np.float32),
                        n=np.array([n_kept]))
    total_elapsed = time.time() - t0
    print('Wrote %s: %d/%d clips (%d skipped silent), %.1f clips/sec overall'
          % (out_path, n_kept, len(records), n_skipped, n_kept / max(total_elapsed, 1e-9)))
    return np.load(out_path)


sub_feats = embed_condition(sub_records, GEN_DIR / 'audio16k', GEN_DIR / 'wavlm_feats.npz', has_k=True)
full_feats = (embed_condition(full_records, FULL_DIR / 'audio16k', FULL_DIR / 'wavlm_feats.npz', has_k=False)
             if full_jobs else None)

## Stage 5: the extended K=64 scaling sweep

Same machinery as `py/phase2a_scaling.py`, reused by import
(`build_xy` for the WavLM/`c_realized` id-matching, `fit_ridge`,
`per_component_r2`, `pooled_r2`, `mean_cosine`, and
`phase2a_ceiling_audit.numerical_rank` / `.oracle_projection` for the
ceiling) -- only the sweep of `n_train` values is extended, since this
corpus's training pool goes well past the 480 that capped the original
sweep. The test split stays fixed across every `n_train` and rep, exactly
as `phase2a_scaling.py` does, so the curve is not confounded by test-set
churn.

The **oracle ceiling is reported at every point** so it is visible exactly
where estimator rank stops binding (it should sit near 1.0 once
`n_train >> K`) and the achieved curve starts measuring what the audio
actually carries rather than how much rank the probe was given.

Phase 2a's own numbers (`py/results/phase2a/scaling.json`, K=64) are
plotted alongside as the low-n baseline this sweep extends -- not
re-measured here, since that would need Phase 2a's original render
corpus, which lives on a different machine and was never committed
(generated assets are gitignored, per CLAUDE.md).

In [ ]:
# 12. Extended n_train sweep at K, using the same fitting/ceiling machinery
# as py/phase2a_scaling.py.
from phase2a_ceiling_audit import numerical_rank, oracle_projection
from phase2b_probe import fit_ridge
from phase2b_subspace_probe import build_xy, mean_cosine, per_component_r2, pooled_r2

LAYERS = [3, 4, 5]     # same default representation as phase2a_scaling.py / phase2b_subspace_probe.py
N_REPS = 5             # reps per n_train, since small n_train is noisy (matches phase2a_scaling.py)

X, Y, rms_pre, itr, ite, n_matched, n_missing = build_xy(K, sub_records, sub_feats,
                                                         np.load(GEN_DIR / 'subspace.npz'), LAYERS)
Xtr_full, Ytr_full = X[itr], Y[itr]
Xte, Yte = X[ite], Y[ite]
n_train_full = len(itr)
print('K=%d: %d matched (%d missing from feats), n_train_full=%d, n_test=%d'
      % (K, n_matched, n_missing, n_train_full, len(ite)))

# log-spaced sweep from a small floor up to the full training pool, always
# including the classic checkpoints (120/240/360/480) for direct comparison
# with py/results/phase2a/scaling.json.
sweep_targets = sorted(set([30, 60, 120, 240, 360, 480] +
                           list(np.unique(np.round(np.geomspace(500, n_train_full, 8)).astype(int)))))
sweep_targets = [n for n in sweep_targets if n <= n_train_full]

sweep_rows = []
for n_train in sweep_targets:
    reps = []
    for rep in range(N_REPS):
        rng = np.random.default_rng(2_000_000 * K + 1_000 * n_train + rep)
        sub = rng.choice(len(Xtr_full), size=n_train, replace=False)
        Xtr, Ytr = Xtr_full[sub], Ytr_full[sub]
        Ypred, alpha = fit_ridge(Xtr, Ytr, Xte)
        comp_r2 = per_component_r2(Yte, Ypred)
        rank_eps, _, _ = numerical_rank(Ytr)
        P_oracle, _ = oracle_projection(Ytr, Yte, rank_eps)
        oracle_r2 = pooled_r2(Ytr, Yte, P_oracle)
        reps.append({'achieved_mean_r2': float(np.nanmean(comp_r2)), 'oracle_r2': oracle_r2,
                     'cosine': mean_cosine(Yte, Ypred)})
    achieved = np.array([r['achieved_mean_r2'] for r in reps])
    oracle = np.array([r['oracle_r2'] for r in reps])
    cosine = np.array([r['cosine'] for r in reps])
    sweep_rows.append({'n_train': n_train, 'n_reps': N_REPS,
                       'achieved_mean_r2_mean': float(achieved.mean()), 'achieved_mean_r2_std': float(achieved.std()),
                       'oracle_r2_mean': float(oracle.mean()), 'cosine_mean': float(cosine.mean())})
    print('n_train=%-6d achieved_R2=%.4f+-%.4f  oracle_R2=%.4f  cosine=%.4f'
          % (n_train, sweep_rows[-1]['achieved_mean_r2_mean'], sweep_rows[-1]['achieved_mean_r2_std'],
             sweep_rows[-1]['oracle_r2_mean'], sweep_rows[-1]['cosine_mean']), flush=True)

clean = [s for s in sweep_rows if s['oracle_r2_mean'] >= 0.999]
if len(clean) >= 2:
    x = np.log([s['n_train'] for s in clean])
    y = [s['achieved_mean_r2_mean'] for s in clean]
    slope, intercept = np.polyfit(x, y, 1)
    r = float(np.corrcoef(x, y)[0, 1])
    print('\nRank-clean log-linear trend: slope=%.4f per log(n), pearson_r=%.4f' % (slope, r))
    print("(Phase 2a's original rank-clean slope at K=64 was 0.163; compare directly.)")
else:
    print('\nFewer than 2 rank-clean points (oracle_r2 >= 0.999) -- n_train_full may still be '
          'too small relative to K for a clean trend fit.')

PHASE2A_ORIGINAL_K64 = [
    {'n_train': 120, 'achieved_mean_r2_mean': 0.0047, 'oracle_r2_mean': 1.0},
    {'n_train': 240, 'achieved_mean_r2_mean': 0.1159, 'oracle_r2_mean': 1.0},
    {'n_train': 360, 'achieved_mean_r2_mean': 0.1813, 'oracle_r2_mean': 1.0},
    {'n_train': 480, 'achieved_mean_r2_mean': 0.2321, 'oracle_r2_mean': 1.0},
]  # py/results/phase2a/scaling.json, K=64 -- plotted for reference, not re-measured here
print("\nPhase 2a original (different machine's corpus, for reference):")
for r in PHASE2A_ORIGINAL_K64:
    print('  n_train=%-6d achieved_R2=%.4f  oracle_R2=%.4f' % (r['n_train'], r['achieved_mean_r2_mean'], r['oracle_r2_mean']))

## Stage 6: the full-space run -- Phase 2b's original question, adequately powered

This is the point of the notebook: no subspace projection, target is the
full 6,144-dim active-row residual (`style_ttl[active_rows] - base`), and
`n_train` is now close to that target's own dimensionality instead of
240/6144 = 3.9%. Same probe machinery as the K-sweep above
(`fit_ridge`, `per_component_r2`, `pooled_r2`, and the oracle ceiling), and
the achieved + oracle numbers are printed next to Phase 2b's original
`eps0.20` result (`py/results/phase2a/ceiling_audit.json`:
`oracle_resid_r2 = 0.0237`, `achieved_resid_r2_best = -0.0169`, both at
`n_train=240`) so the improvement -- or lack of one -- is a direct,
apples-to-apples comparison rather than a new number in isolation.

Runs only if `GENERATE_FULL_SPACE` was set above.

In [ ]:
# 13. Full-space probe: WavLM -> full 6,144-dim active-row residual.
if not GENERATE_FULL_SPACE:
    print('GENERATE_FULL_SPACE is False -- nothing to run here. Set it and rerun stages 3-4 to '
          'get the full-space corpus and features first.')
    full_result = None
else:
    from phase2b_probe import shuffled_targets

    full_styles = np.load(FULL_DIR / 'styles.npz')
    ttl_full = full_styles['ttl']            # (N_FULL, 50, 256)
    base_active = full_styles['base_ttl'][ACTIVE_ROWS, :].reshape(-1).astype(np.float64)
    Y_full_all = (ttl_full[:, ACTIVE_ROWS, :].reshape(len(ttl_full), -1).astype(np.float64)
                 - base_active)   # within-family residual, matches phase2b_probe's target

    key_to_row = {int(i): n for n, i in enumerate(full_feats['idx'])}
    kept = [r for r in full_records if int(r['idx']) in key_to_row]
    feat_rows = np.array([key_to_row[int(r['idx'])] for r in kept])
    from phase2b_wavlm import rep_from
    Xf = rep_from(full_feats['mean'][feat_rows], full_feats['std'][feat_rows], LAYERS, 'meanstd')
    Yf = Y_full_all[[int(r['idx']) for r in kept]]
    rms_pre_f = full_feats['rms_pre'][feat_rows].astype(np.float64)
    split_f = np.array([r['split'] for r in kept])
    itr_f, ite_f = np.where(split_f == 'train')[0], np.where(split_f == 'test')[0]
    print('full-space: %d matched clips (%d missing from feats), n_train=%d, n_test=%d, target_dim=%d'
          % (len(kept), len(full_records) - len(kept), len(itr_f), len(ite_f), Yf.shape[1]))

    # ---- achieved: ridge probe, WavLM -> full-space residual ----
    Ypred_f, alpha_f = fit_ridge(Xf[itr_f], Yf[itr_f], Xf[ite_f])
    achieved_r2 = pooled_r2(Yf[itr_f], Yf[ite_f], Ypred_f)
    cos_f = mean_cosine(Yf[ite_f], Ypred_f)

    # ---- out-of-sample oracle ceiling at this n_train, exactly as
    # phase2a_ceiling_audit.py computes it: orthogonal projection of the test
    # targets onto row_space(train targets). This is the load-bearing
    # control -- it tells us whether the run below is still rank-limited. ----
    rank_eps_f, rank_loose_f, s_f = numerical_rank(Yf[itr_f])
    P_oracle_f, _ = oracle_projection(Yf[itr_f], Yf[ite_f], rank_eps_f)
    oracle_r2_f = pooled_r2(Yf[itr_f], Yf[ite_f], P_oracle_f)
    target_dim_f = Yf.shape[1]
    predicted_ceiling_frac = min(1.0, len(itr_f) / target_dim_f)

    # ---- Hewitt & Liang shuffled-target control: identical ridge, targets
    # randomly re-paired across samples. Must land near 0, following
    # phase2b_subspace_probe.py's pattern. ----
    Yc_f = shuffled_targets(Yf, np.arange(len(Yf)))
    Ypred_fc, alpha_fc = fit_ridge(Xf[itr_f], Yc_f[itr_f], Xf[ite_f])
    shuffled_r2_f = pooled_r2(Yc_f[itr_f], Yc_f[ite_f], Ypred_fc)

    # ---- loudness control: predict the target from pre-normalization clip
    # RMS alone (a 1-dim feature). Must be near 0, or the achieved result
    # above is confounded by perturbation magnitude buying plain loudness. ----
    Xrms_tr_f = rms_pre_f[itr_f].reshape(-1, 1)
    Xrms_te_f = rms_pre_f[ite_f].reshape(-1, 1)
    Ypred_rms_f, alpha_rms_f = fit_ridge(Xrms_tr_f, Yf[itr_f], Xrms_te_f)
    loudness_r2_f = pooled_r2(Yf[itr_f], Yf[ite_f], Ypred_rms_f)

    # ---- verdict: rank-limited vs genuinely measuring audio recoverability.
    # Thresholds:
    #   CEILING_NEAR_ONE (0.95)   -- oracle ceiling counts as "near 1" above this
    #   NEAR_CEILING_FRAC (0.8)   -- achieved counts as "tracking the ceiling"
    #                                once it reaches this fraction of the ceiling
    #   SHUFFLED_MARGIN (0.05)    -- achieved must clear the shuffled control by
    #                                at least this much R^2 to call it "meaningfully above"
    #   LOUDNESS_CONFOUND (0.05)  -- loudness control above this flags a confound
    #                                regardless of which regime the run lands in
    CEILING_NEAR_ONE = 0.95
    NEAR_CEILING_FRAC = 0.8
    SHUFFLED_MARGIN = 0.05
    LOUDNESS_CONFOUND = 0.05

    if oracle_r2_f < CEILING_NEAR_ONE and achieved_r2 >= NEAR_CEILING_FRAC * oracle_r2_f:
        regime = 'RANK-LIMITED'
        verdict = ('RANK-LIMITED: oracle ceiling (%.4f) is well below 1 and the achieved R^2 '
                   '(%.4f) sits at or near that ceiling -- this run is still measuring how much '
                   'rank n_train=%d buys against a %d-dim target, not what the audio carries. '
                   'Needs more n_train (or a smaller target, i.e. subspace projection) before '
                   'the achieved number means anything.'
                   % (oracle_r2_f, achieved_r2, len(itr_f), target_dim_f))
    elif oracle_r2_f >= CEILING_NEAR_ONE and (achieved_r2 - shuffled_r2_f) >= SHUFFLED_MARGIN:
        regime = 'MEASURING AUDIO'
        verdict = ('MEASURING AUDIO RECOVERABILITY: oracle ceiling (%.4f) is near 1, so rank is '
                   'no longer binding, and achieved R^2 (%.4f) clears the shuffled-target control '
                   '(%.4f) by %.4f -- this is evidence about what the audio carries, not an '
                   'artifact of estimator rank.' % (oracle_r2_f, achieved_r2, shuffled_r2_f,
                                                     achieved_r2 - shuffled_r2_f))
    else:
        regime = 'INCONCLUSIVE'
        verdict = ('INCONCLUSIVE: neither the rank-limited nor the audio-recoverable pattern is '
                   'clearly met (oracle ceiling=%.4f, achieved=%.4f, shuffled=%.4f) -- more '
                   'n_train is needed before drawing either conclusion.'
                   % (oracle_r2_f, achieved_r2, shuffled_r2_f))
    if loudness_r2_f > LOUDNESS_CONFOUND:
        verdict += (' CAVEAT: loudness control R^2=%.4f exceeds the %.2f confound threshold -- '
                    'part of the achieved number may be plain perturbation-magnitude loudness, '
                    'not a subtler audio effect.' % (loudness_r2_f, LOUDNESS_CONFOUND))

    full_result = {
        'n_train': int(len(itr_f)), 'n_test': int(len(ite_f)), 'target_dim': int(target_dim_f),
        'achieved_r2_testmean_baseline': achieved_r2, 'oracle_r2_testmean_baseline': oracle_r2_f,
        'oracle_numerical_rank': rank_eps_f, 'mean_cosine_pred_vs_true': cos_f,
        'ridge_alpha': alpha_f, 'layers_1based': LAYERS,
        'predicted_ceiling_fraction_ntrain_over_dims': predicted_ceiling_frac,
        'shuffled_target_control_r2': shuffled_r2_f,
        'loudness_control_r2': loudness_r2_f, 'loudness_control_alpha': alpha_rms_f,
        'regime': regime, 'verdict': verdict,
    }

    hdr = '%14s%14s%12s%12s' % ('achieved_R2', 'oracle_R2', 'shuffled', 'loudness')
    print('\nFull-space (this corpus): n_train=%d, target_dim=%d' % (full_result['n_train'], target_dim_f))
    print(hdr)
    print('-' * len(hdr))
    print('%14.4f%14.4f%12.4f%12.4f' % (achieved_r2, oracle_r2_f, shuffled_r2_f, loudness_r2_f))
    print('\noracle rank=%d, cosine=%.4f, predicted ceiling n_train/target_dim=%.4f (measured %.4f)'
          % (rank_eps_f, cos_f, predicted_ceiling_frac, oracle_r2_f))
    if len(itr_f) < target_dim_f:
        print('NOTE: n_train (%d) is below target_dim (%d) -- the oracle ceiling is expected to '
              'sit near n_train/target_dim = %.4f rather than 1.0, and that is a rank limit, not '
              'an audio limit.' % (len(itr_f), target_dim_f, predicted_ceiling_frac))
    print('\nPhase 2b original (py/results/phase2a/ceiling_audit.json, eps0.20, n_train=240):')
    print('  achieved R^2 = -0.0169   oracle ceiling R^2 = 0.0237 (rank 240)')
    print('\nThe oracle ceiling moving from 0.024 toward 1.0 is the mechanical part -- more '
          'training rows span more of a 6,144-dim target. Whether the ACHIEVED number moves with '
          'it is the actual finding: audio carrying the perturbation, at the scale Phase 2b asked '
          'about in the first place, rather than only inside an artificially shrunk subspace.')
    print('\nVERDICT: %s' % verdict)

## Stage 7: save and summarize

Everything measured above -- the config used, the extended K sweep, and the
full-space result next to Phase 2b's original -- goes into one JSON on
Drive, plus a plain-text summary table.

In [ ]:
# 14. Save results and print the summary table.
summary = {
    'experiment': 'phase2a_scaling_colab',
    'generated_at': datetime.datetime.now(datetime.timezone.utc).isoformat(),
    'config': {
        'N_TOTAL': N_TOTAL, 'K': K, 'EPS': EPS, 'TEST_FRAC': TEST_FRAC,
        'GENERATE_FULL_SPACE': GENERATE_FULL_SPACE, 'N_FULL': N_FULL if GENERATE_FULL_SPACE else None,
        'N_WORKERS': N_WORKERS, 'layers_1based': LAYERS,
    },
    'generation_timing': {
        'measured_s_per_clip_this_instance': this_machine_s_per_clip,
        'used_gpu_for_generation': USE_GPU_FOR_GENERATION,
        'reference_s_per_clip_other_machine': REFERENCE_S_PER_CLIP,
    },
    'k_sweep': {'K': K, 'n_train_full': n_train_full, 'n_test': int(len(ite)), 'sweep': sweep_rows},
    'phase2a_original_k64_reference': PHASE2A_ORIGINAL_K64,
    'full_space_result': full_result,
    'phase2b_original_reference': {
        'n_train': 240, 'achieved_r2': -0.0169, 'oracle_r2': 0.0237,
        'source': 'py/results/phase2a/ceiling_audit.json (eps0.20)',
    },
}
with open(RESULTS / 'phase2a_scaling_colab_results.json', 'w') as f:
    _json.dump(summary, f, indent=2)
print('Wrote %s\n' % (RESULTS / 'phase2a_scaling_colab_results.json'))

print('=' * 78)
print('SUMMARY')
print('=' * 78)
print('\nK=%d sweep (this corpus, n_train_full=%d):' % (K, n_train_full))
print('%10s %14s %14s %10s' % ('n_train', 'achieved_R2', 'oracle_R2', 'cosine'))
for s in sweep_rows:
    print('%10d %14.4f %14.4f %10.4f' % (s['n_train'], s['achieved_mean_r2_mean'],
                                         s['oracle_r2_mean'], s['cosine_mean']))
print('\nFor reference, Phase 2a\'s original K=64 sweep (different machine\'s corpus):')
for r in PHASE2A_ORIGINAL_K64:
    print('%10d %14.4f %14.4f' % (r['n_train'], r['achieved_mean_r2_mean'], r['oracle_r2_mean']))

if full_result is not None:
    print('\nFull-space (6,144-dim, unrestricted) probe:')
    print('  this corpus:  n_train=%-6d achieved_R2=%.4f  oracle_R2=%.4f  cosine=%.4f'
          % (full_result['n_train'], full_result['achieved_r2_testmean_baseline'],
             full_result['oracle_r2_testmean_baseline'], full_result['mean_cosine_pred_vs_true']))
    print('  Phase 2b original: n_train=240      achieved_R2=-0.0169  oracle_R2=0.0237')
else:
    print('\nFull-space probe: not run (GENERATE_FULL_SPACE was False).')

## What this notebook does and does not settle

- **It settles** whether the K=64 recovery curve keeps rising past
  n_train=480, and by how much, with the oracle ceiling reported at every
  point so any flattening is legible rather than assumed. Read the sweep
  table above, not just the printed slope, before concluding anything --
  a slope is a summary of a curve, not the curve.
- **It settles**, for the first time with adequate power, whether the
  *unrestricted* 6,144-dim active-row residual is linearly recoverable
  from audio at all -- Phase 2b's original question. The full-space cell
  above now runs the same controls `phase2b_subspace_probe.py` established
  for the K-subspace runs: a Hewitt & Liang shuffled-target refit, a
  loudness-only (`rms_pre`) refit, and the out-of-sample oracle ceiling at
  the actual n_train reached, with a printed verdict that names the regime
  (rank-limited vs. measuring audio recoverability) rather than leaving
  that judgment to the reader. If the achieved R^2 lands in the
  audio-recoverable regime -- oracle ceiling near 1, achieved meaningfully
  above both the shuffled and loudness controls -- that reopens the
  amortized-style-inversion route Phase 2b closed.
- **It does not settle** whether recovery saturates at some n_train beyond
  what this notebook generated -- only that it does or does not by the
  corpus size chosen here. If the curve is still rising at the top of the
  sweep, the honest conclusion is "still rising," not a new extrapolated
  ceiling.
- **It does not by itself judge audibility.** A positive R^2 here is a
  statement about a WavLM embedding and a ridge probe, not about what a
  listener hears. Per CLAUDE.md, any claim built on this notebook's
  numbers that reaches a listening decision still needs a listening bench,
  spectrogram diffs on aligned/level-matched clips, and calibration of any
  distance metric against known-same/known-different pairs before citing
  it as evidence.

In [ ]:
# 15. Bundle results (not audio -- 8,000+ clips is tens of GB and gitignored
# in this repo anyway; keep the WAVs on Drive, take the numbers with you).
import shutil

from google.colab import files

bundle = Path('/content/phase2a-scaling-results')
if bundle.exists():
    shutil.rmtree(bundle)
bundle.mkdir()
shutil.copy2(RESULTS / 'phase2a_scaling_colab_results.json', bundle / 'results.json')
shutil.copy2(GEN_DIR / 'manifest.json', bundle / 'subspace_manifest.json')
if (FULL_DIR / 'manifest.json').exists():
    shutil.copy2(FULL_DIR / 'manifest.json', bundle / 'fullspace_manifest.json')

archive = shutil.make_archive('/content/phase2a-scaling-results', 'zip', str(bundle))
print('Archive: %s (%.1f MB)' % (archive, Path(archive).stat().st_size / 1e6))
print('Drive copy of everything (including audio): %s' % WORKSPACE)
files.download(archive)